[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [APIs and JSON](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)

# Validating Requests &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The cell below rebuilds what the notebook set up: the practice API, whose `serve` runs each app,
FastAPI and Pydantic. Run it first, then the tasks in order, since tasks 2, 3, 4 and 6 build on the
model before them.


In [1]:
import importlib
import sys
import urllib.request
from pathlib import Path
from typing import Annotated

import requests
from fastapi import FastAPI, Query
from fastapi.exceptions import RequestValidationError
from fastapi.responses import JSONResponse
from pydantic import BaseModel, ConfigDict, Field, field_validator

PRACTICE_API = "https://raw.githubusercontent.com/johnfisher-ai/Python-Visual-Guides/main/notebooks/apis-and-json/practice_api.py"

if "google.colab" in sys.modules or not Path("practice_api.py").exists():
    urllib.request.urlretrieve(PRACTICE_API, "practice_api.py")    # in Colab, on every run

import practice_api
importlib.reload(practice_api)    # runs the file as it is now, not a copy imported earlier

BASE = practice_api.start()
print("ready:", BASE)


ready: http://127.0.0.1:8765


**1.** A body as a model.


In [2]:
class Reading(BaseModel):
    station: str
    temperature_c: float


app = FastAPI()


@app.post("/readings", status_code=201)
def add_reading(reading: Reading):
    return reading


app_url = practice_api.serve(app)
response = requests.post(f"{app_url}/readings", json={"station": "oslo", "temperature_c": -4.2}, timeout=10)
print(response.status_code, response.json())


201 {'station': 'oslo', 'temperature_c': -4.2}


The route returned the model, and FastAPI sent its fields as JSON, with `201` from the decorator.


**2.** A rule for a field.


In [3]:
class Reading(BaseModel):
    station: str
    temperature_c: float = Field(ge=-90, le=60)


app = FastAPI()


@app.post("/readings", status_code=201)
def add_reading(reading: Reading):
    return reading


app_url = practice_api.serve(app)
response = requests.post(f"{app_url}/readings", json={"station": "oslo", "temperature_c": 75}, timeout=10)
print(response.status_code)
for problem in response.json()["detail"]:
    print(problem["loc"], problem["msg"])


422
['body', 'temperature_c'] Input should be less than or equal to 60


-90 and 60 are the limits that the **Exceptions as Classes** notebook in the **Object-Oriented
Python** guide gave a reading that could have been measured on Earth.


**3.** Strict, with no fields the model does not declare.


In [4]:
class Reading(BaseModel):
    model_config = ConfigDict(extra="forbid")

    station: str
    temperature_c: float = Field(ge=-90, le=60, strict=True)


app = FastAPI()


@app.post("/readings", status_code=201)
def add_reading(reading: Reading):
    return reading


app_url = practice_api.serve(app)
response = requests.post(f"{app_url}/readings", json={"station": "oslo", "temperature_c": "-4.2", "unit": "C"}, timeout=10)
print(response.status_code)
for problem in response.json()["detail"]:
    print(problem["loc"], problem["msg"])


422
['body', 'temperature_c'] Input should be a valid number
['body', 'unit'] Extra inputs are not permitted


Both problems in one answer: the temperature is text, and `unit` is not a field of `Reading`.


**4.** A rule of your own.


In [5]:
class Reading(BaseModel):
    model_config = ConfigDict(extra="forbid")

    station: str
    temperature_c: float = Field(ge=-90, le=60, strict=True)

    @field_validator("station")
    @classmethod
    def known_station(cls, station):
        station = station.lower()
        if station not in practice_api.STATIONS:
            raise ValueError(f"no station has the id {station!r}")
        return station


app = FastAPI()


@app.post("/readings", status_code=201)
def add_reading(reading: Reading):
    return reading


app_url = practice_api.serve(app)
for station in ["Oslo", "narvik"]:
    response = requests.post(f"{app_url}/readings", json={"station": station, "temperature_c": -4.2}, timeout=10)
    print(response.status_code, response.json())


201 {'station': 'oslo', 'temperature_c': -4.2}
422 {'detail': [{'type': 'value_error', 'loc': ['body', 'station'], 'msg': "Value error, no station has the id 'narvik'", 'input': 'narvik', 'ctx': {'error': {}}}]}


`Oslo` came back as `oslo`, the value the validator returned, and `narvik` was refused with the
validator's message after `Value error, `.


**5.** A rule for a query parameter.


In [6]:
app = FastAPI()


@app.get("/readings")
def readings(hours: Annotated[int, Query(ge=1, le=72)] = 24):
    return {"hours": hours}


app_url = practice_api.serve(app)
response = requests.get(f"{app_url}/readings?hours=0", timeout=10)
print(response.status_code, response.json()["detail"][0]["msg"])


422 Input should be greater than or equal to 1


The practice API holds 72 hours of readings, so a limit of 72 is every reading it has.


**6.** A 422 in a shape of your own.


In [7]:
app = FastAPI()


@app.exception_handler(RequestValidationError)
def reading_problems(request, error):
    problems = [{"field": ".".join(str(part) for part in problem["loc"][1:]) or None, "problem": problem["msg"]}
                for problem in error.errors()]
    return JSONResponse(status_code=422, content={"error": "the reading has problems", "problems": problems})


@app.post("/readings", status_code=201)
def add_reading(reading: Reading):
    return reading


app_url = practice_api.serve(app)
response = requests.post(f"{app_url}/readings", json={"station": "oslo", "temperature_c": "-4.2", "unit": "C"}, timeout=10)
print(response.status_code, response.json())


422 {'error': 'the reading has problems', 'problems': [{'field': 'temperature_c', 'problem': 'Input should be a valid number'}, {'field': 'unit', 'problem': 'Extra inputs are not permitted'}]}


The handler and the route are made in the same cell, before the app answers anything, so the handler
answers the `422`, with the two problems from task 3 in the practice API's shape.


---

&#8592; **Back to:** [Validating Requests](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/16-validating-requests.ipynb)  &nbsp;&middot;&nbsp;  [APIs and JSON Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)
